# Comparative Evaluation: BART vs. T5 on CNN/DailyMail
**Course:** CSCI 390, Fisk University  
**Author:** Aayush Khanal  

This notebook trains and evaluates both `facebook/bart-base` and `t5-base` under strictly matched conditions on Kaggle's free T4 GPU (16 GB).

## 1. Verify GPU Hardware

In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Clone Repository and Install Dependencies

In [ ]:
import os

# Clone the repository if not already cloned
if not os.path.exists("text-summarizer-bart-t5"):
    !git clone https://github.com/Aayush30197/text-summarizer-bart-t5.git
    %cd text-summarizer-bart-t5
else:
    %cd text-summarizer-bart-t5
    !git pull origin main

# Install pinned dependencies
!pip install -q -r requirements.txt

## 3. Fine-Tune BART (`facebook/bart-base`)
- 12,000 CNN/DailyMail articles (1 epoch, ~375 steps)
- Precision: fp16 on T4 GPU
- Auto-saves checkpoints every 100 steps

In [ ]:
!python src/train.py --model bart

## 4. Fine-Tune T5 (`t5-base`)
- 12,000 CNN/DailyMail articles (1 epoch, ~375 steps)
- Input prefix: `summarize: `
- Precision: fp32 safe mode (avoids NaN loss on T4)

In [ ]:
!python src/train.py --model t5

## 5. Evaluate Fine-Tuned Checkpoints (Test Set: 1,000 articles)
Measures ROUGE-1/2/L, generation latency, and peak memory under fixed decoding parameters (`num_beams: 4`, `min_length: 30`, `max_length: 128`).

In [ ]:
# Evaluate BART
!python src/eval_model.py --model bart

# Evaluate T5
!python src/eval_model.py --model t5

## 6. Archive Results and Best Checkpoints for Download

In [ ]:
# Create an archive of evaluation results and trained models in /kaggle/working
!tar -czvf /kaggle/working/experiment_results_and_checkpoints.tar.gz results/ checkpoints/best_* 2>/dev/null || tar -czvf /kaggle/working/experiment_results.tar.gz results/